# Customer Support Intelligence Platform — Phase 2: Text Preprocessing & Feature Engineering

**Brief requirements covered (Day 5-6):**
- Clean Ticket Subject and Ticket Description (lowercase, remove punctuation/HTML,
  tokenize, remove stopwords, lemmatize)
- Compute TF-IDF vectors
- Compute word embeddings (Word2Vec)
- Extract text length, word count, and sentiment polarity as features
- Encode categorical columns (Ticket Channel, Product Purchased, Customer Gender)
- Parse Date of Purchase for tenure features

**Design choice:** text cleaning logic lives in `src/preprocessing.py`, not copied
into this notebook - the brief requires reusable code, and this guarantees the
exact same cleaning runs at training time and later at API inference time.

In [3]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
import numpy as np
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from gensim.models import Word2Vec
from textblob import TextBlob

from preprocessing import clean_text, get_text_stats

df = pd.read_csv("../data_raw/customer_support_tickets.csv")
print(f"Loaded {len(df)} rows")

Loaded 8469 rows


## 1. Text cleaning

Applying `clean_text()` from `src/preprocessing.py` to both Ticket Subject and
Ticket Description. This is the exact same function the deployed API will use
later, so predictions on new tickets are preprocessed identically to training data.

In [4]:
df["Subject_Clean"] = df["Ticket Subject"].apply(clean_text)
df["Description_Clean"] = df["Ticket Description"].apply(clean_text)

# Combined text field for the DistilBERT fine-tuning step later (brief specifies
# "combined Ticket Subject + Ticket Description text" for that task)
df["Combined_Text_Clean"] = df["Subject_Clean"] + " " + df["Description_Clean"]

print("Before cleaning:")
print(df["Ticket Description"].iloc[0][:200])
print("\nAfter cleaning:")
print(df["Description_Clean"].iloc[0][:200])

Before cleaning:
I'm having an issue with the {product_purchased}. Please assist.

Your billing zip code is: 71701.

We appreciate that you have requested a website address.

Please double check your email address. I'

After cleaning:
im issue productpurchased please assist billing zip code appreciate requested website address please double check email address ive tried troubleshooting step mentioned user manual issue persists


## 2. Text length and word count features

Computed on the ORIGINAL (uncleaned) text, not the cleaned version - cleaning
strips stopwords/punctuation, which would understate how much a customer
actually wrote.

In [5]:
stats = df["Ticket Description"].apply(get_text_stats)
df["Description_Char_Count"] = stats.apply(lambda x: x["char_count"])
df["Description_Word_Count"] = stats.apply(lambda x: x["word_count"])

print(df[["Description_Char_Count", "Description_Word_Count"]].describe())

       Description_Char_Count  Description_Word_Count
count             8469.000000             8469.000000
mean               289.821939               46.467352
std                 43.593954                8.461730
min                151.000000               21.000000
25%                273.000000               43.000000
50%                298.000000               49.000000
75%                318.000000               52.000000
max                397.000000               63.000000


## 3. Sentiment polarity

Using TextBlob's polarity score (-1 = very negative, +1 = very positive) on the
original ticket description. This could be a genuinely useful signal for
priority prediction - an angry-sounding ticket might correlate with higher
urgency, independent of its topic category.

In [6]:
def get_sentiment_polarity(text):
    if not isinstance(text, str) or not text.strip():
        return 0.0
    return TextBlob(text).sentiment.polarity

df["Sentiment_Polarity"] = df["Ticket Description"].apply(get_sentiment_polarity)

print(df["Sentiment_Polarity"].describe())
print(f"\nMean sentiment by Ticket Priority:")
print(df.groupby("Ticket Priority")["Sentiment_Polarity"].mean().sort_values())
print("\n(If Critical tickets don't show notably more negative sentiment than Low,")
print("that's consistent with our earlier finding that this dataset's text content")
print("doesn't strongly differentiate ticket categories - worth noting either way.)")

count    8469.000000
mean        0.059683
std         0.201524
min        -0.750000
25%        -0.033333
50%         0.020000
75%         0.161932
max         1.000000
Name: Sentiment_Polarity, dtype: float64

Mean sentiment by Ticket Priority:
Ticket Priority
Medium      0.051535
Low         0.059846
Critical    0.063112
High        0.064587
Name: Sentiment_Polarity, dtype: float64

(If Critical tickets don't show notably more negative sentiment than Low,
that's consistent with our earlier finding that this dataset's text content
doesn't strongly differentiate ticket categories - worth noting either way.)


## 4. TF-IDF vectorization

Fit on the cleaned combined text. Saved via joblib so the exact same fitted
vectorizer (not a newly-fit one) is used for baseline models and later at
inference time - refitting would produce different vocabulary/weights.

In [7]:
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
tfidf_matrix = tfidf.fit_transform(df["Combined_Text_Clean"])

print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
print(f"Vocabulary size: {len(tfidf.vocabulary_)}")

import os
os.makedirs("../models", exist_ok=True)
joblib.dump(tfidf, "../models/tfidf_vectorizer.joblib")
print("Saved TF-IDF vectorizer to models/tfidf_vectorizer.joblib")

TF-IDF matrix shape: (8469, 5000)
Vocabulary size: 5000
Saved TF-IDF vectorizer to models/tfidf_vectorizer.joblib


## 5. Word2Vec embeddings

The brief allows either Word2Vec or GloVe. We train Word2Vec directly on our
own corpus (via gensim) rather than downloading external GloVe vectors here -
this is self-contained and doesn't require a separate multi-hundred-MB
download. (Pre-trained GloVe embeddings are used later for the BiLSTM model
in Day 11, per the brief's specific instruction for that step.)

In [8]:
tokenized_corpus = df["Combined_Text_Clean"].apply(str.split).tolist()

w2v_model = Word2Vec(
    sentences=tokenized_corpus,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    seed=42,
)

print(f"Word2Vec vocabulary size: {len(w2v_model.wv)}")

# Sanity check - similar words should cluster together if training worked
if "issue" in w2v_model.wv:
    print("\nWords most similar to 'issue':")
    print(w2v_model.wv.most_similar("issue", topn=5))

w2v_model.save("../models/word2vec_model.model")
print("\nSaved Word2Vec model to models/word2vec_model.model")

Word2Vec vocabulary size: 2651

Words most similar to 'issue':
[('div', 0.4978443384170532), ('rpi', 0.48621129989624023), ('productamount', 0.47251152992248535), ('expert', 0.4701693654060364), ('droid', 0.4451718032360077)]

Saved Word2Vec model to models/word2vec_model.model


## 6. Categorical feature encoding

Brief specifies: encode Ticket Channel, Product Purchased, Customer Gender.
Using Label Encoding here since these will feed into tree-based models
(Random Forest, XGBoost, LightGBM) later, which handle label-encoded
categoricals natively and efficiently - one-hot encoding Product Purchased
specifically would create an excessive number of sparse columns given how
many distinct products exist.

In [9]:
categorical_cols = ["Ticket Channel", "Product Purchased", "Customer Gender"]
encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    df[f"{col}_Encoded"] = le.fit_transform(df[col].astype(str))
    encoders[col] = le
    print(f"{col}: {len(le.classes_)} unique categories")

joblib.dump(encoders, "../models/label_encoders.joblib")
print("\nSaved label encoders to models/label_encoders.joblib")

Ticket Channel: 4 unique categories
Product Purchased: 42 unique categories
Customer Gender: 3 unique categories

Saved label encoders to models/label_encoders.joblib


## 7. Date of Purchase — tenure feature

**Design decision:** the brief asks to "extract ticket age (days since
purchase)" but doesn't specify what reference date to measure "since" from
(there's no explicit "ticket created" timestamp column, only Date of
Purchase and First Response Time). We use the dataset's most recent
timestamp as a fixed reference point, giving a consistent, reproducible
"customer tenure in days" feature.

In [10]:
df["Date of Purchase"] = pd.to_datetime(df["Date of Purchase"])

reference_date = df["Date of Purchase"].max()
df["Customer_Tenure_Days"] = (reference_date - df["Date of Purchase"]).dt.days

print(f"Reference date used: {reference_date}")
print(df["Customer_Tenure_Days"].describe())

Reference date used: 2021-12-30 00:00:00
count    8469.000000
mean      364.933876
std       211.243086
min         0.000000
25%       182.000000
50%       364.000000
75%       546.000000
max       729.000000
Name: Customer_Tenure_Days, dtype: float64


## 8. Save processed dataset

Per the brief's data handling rule: "Never overwrite raw data; keep the
original CSV intact." Saving all engineered features to a NEW file, leaving
`data_raw/customer_support_tickets.csv` completely untouched.

In [11]:
os.makedirs("../data_processed", exist_ok=True)
df.to_csv("../data_processed/tickets_with_features.csv", index=False)

print(f"Saved processed dataset: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"New engineered columns added: {df.shape[1] - 17}")
print("\nOriginal raw CSV at data_raw/customer_support_tickets.csv remains untouched.")

Saved processed dataset: 8469 rows x 27 columns
New engineered columns added: 10

Original raw CSV at data_raw/customer_support_tickets.csv remains untouched.
